<a href="https://colab.research.google.com/github/sanadv/CS360/blob/main/CS360_B%2BTrees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


from __future__ import annotations
from bisect import bisect_left, bisect_right
from typing import Any, Iterator, Optional


class _Node:
    """A single node. Leaves carry parallel `values` and a `next` link;
    internal nodes carry `children` and no values."""
    __slots__ = ("leaf", "keys", "values", "children", "next")

    def __init__(self, leaf: bool):
        self.leaf = leaf
        self.keys: list = []
        self.values: list = []          # used only when leaf
        self.children: list["_Node"] = []  # used only when internal
        self.next: Optional["_Node"] = None  # leaf link (leaves only)


class BPlusTree:
    """A B+ tree mapping ordered keys to values.

    max_keys is the maximum number of keys a node may hold (a node overflows
    at max_keys + 1). With max_keys = 3 the tree matches the slides.
    """

    def __init__(self, max_keys: int = 3):
        if max_keys < 2:
            raise ValueError("max_keys must be >= 2")
        self.max_keys = max_keys
        # Minimum occupancy for NON-root nodes ("half-full", slide 5).
        self.min_leaf = (max_keys + 1) // 2                 # e.g. 3 -> 2
        self.min_internal = (max_keys + 2) // 2 - 1         # e.g. 3 -> 1 (min children 2)
        self.root = _Node(leaf=True)
        self._size = 0

    # ------------------------------------------------------------------ #
    # SEARCH  (slide 6: search always ends at a leaf)                     #
    # ------------------------------------------------------------------ #
    def _find_leaf(self, key) -> _Node:
        """Descend from the root to the leaf that would contain `key`."""
        node = self.root
        while not node.leaf:
            # "equals go right": follow child index = #separators <= key.
            i = bisect_right(node.keys, key)
            node = node.children[i]
        return node

    def search(self, key) -> Optional[Any]:
        """Return the value stored for `key`, or None if absent."""
        leaf = self._find_leaf(key)
        i = bisect_left(leaf.keys, key)
        if i < len(leaf.keys) and leaf.keys[i] == key:
            return leaf.values[i]
        return None

    def __contains__(self, key) -> bool:
        leaf = self._find_leaf(key)
        i = bisect_left(leaf.keys, key)
        return i < len(leaf.keys) and leaf.keys[i] == key

    # ------------------------------------------------------------------ #
    # RANGE SCAN  (slide 7: find the start, then walk the leaf links)     #
    # ------------------------------------------------------------------ #
    def range(self, lo, hi) -> Iterator[tuple]:
        """Yield (key, value) for every key with lo <= key <= hi, in order.
        One descent to the first leaf, then a sequential walk of leaf links."""
        node = self._find_leaf(lo)
        i = bisect_left(node.keys, lo)
        while node is not None:
            while i < len(node.keys):
                k = node.keys[i]
                if k > hi:
                    return
                yield (k, node.values[i])
                i += 1
            node = node.next   # follow the leaf link
            i = 0

    def items(self) -> Iterator[tuple]:
        """Yield every (key, value) in sorted order via the leaf chain."""
        node = self.root
        while not node.leaf:
            node = node.children[0]
        while node is not None:
            for k, v in zip(node.keys, node.values):
                yield (k, v)
            node = node.next

    # ------------------------------------------------------------------ #
    # INSERTION  (slide 8: copy-up on leaf split, move-up on internal)    #
    # ------------------------------------------------------------------ #
    def insert(self, key, value: Any = None) -> None:
        """Insert key->value. If key exists, its value is updated in place."""
        if value is None:
            value = key
        split = self._insert(self.root, key, value)
        if split is not None:
            sep, left, right = split
            new_root = _Node(leaf=False)
            new_root.keys = [sep]
            new_root.children = [left, right]
            self.root = new_root

    def _insert(self, node: _Node, key, value):
        """Insert into subtree `node`. Returns (sep, left, right) if `node`
        split and the parent must absorb `sep`, else None."""
        if node.leaf:
            i = bisect_left(node.keys, key)
            if i < len(node.keys) and node.keys[i] == key:
                node.values[i] = value          # update existing key
                return None
            node.keys.insert(i, key)
            node.values.insert(i, value)
            self._size += 1
            if len(node.keys) > self.max_keys:
                return self._split_leaf(node)
            return None

        # internal: route down, then absorb any split coming back up
        i = bisect_right(node.keys, key)
        split = self._insert(node.children[i], key, value)
        if split is None:
            return None
        sep, left, right = split
        node.keys.insert(i, sep)
        node.children[i] = left
        node.children.insert(i + 1, right)
        if len(node.keys) > self.max_keys:
            return self._split_internal(node)
        return None

    def _split_leaf(self, node: _Node):
        """Split an overflowed leaf; COPY the right half's first key up."""
        mid = len(node.keys) // 2
        right = _Node(leaf=True)
        right.keys = node.keys[mid:]
        right.values = node.values[mid:]
        node.keys = node.keys[:mid]
        node.values = node.values[:mid]
        # re-link the leaf chain: node -> right -> (old node.next)
        right.next = node.next
        node.next = right
        sep = right.keys[0]        # COPY up (the key also stays below)
        return (sep, node, right)

    def _split_internal(self, node: _Node):
        """Split an overflowed internal node; MOVE the middle key up."""
        mid = len(node.keys) // 2
        sep = node.keys[mid]       # MOVE up (does NOT stay at this level)
        right = _Node(leaf=False)
        right.keys = node.keys[mid + 1:]
        right.children = node.children[mid + 1:]
        node.keys = node.keys[:mid]
        node.children = node.children[:mid + 1]
        return (sep, node, right)

    # ------------------------------------------------------------------ #
    # DELETION  (slide 16: delete from a leaf; borrow or merge on under-  #
    # flow; separators may linger as pure routing guides)                #
    # ------------------------------------------------------------------ #
    def delete(self, key) -> bool:
        """Remove `key`. Returns True if it was present."""
        removed = self._delete(self.root, key)
        # Collapse a root that has thinned to a single child.
        if not self.root.leaf and len(self.root.keys) == 0:
            self.root = self.root.children[0]
        return removed

    def _delete(self, node: _Node, key) -> bool:
        if node.leaf:
            i = bisect_left(node.keys, key)
            if i < len(node.keys) and node.keys[i] == key:
                node.keys.pop(i)
                node.values.pop(i)
                self._size -= 1
                return True
            return False

        i = bisect_right(node.keys, key)
        removed = self._delete(node.children[i], key)
        if removed and self._underflow(node.children[i]):
            self._fix_underflow(node, i)
        return removed

    def _underflow(self, node: _Node) -> bool:
        if node.leaf:
            return len(node.keys) < self.min_leaf
        return len(node.keys) < self.min_internal

    def _fix_underflow(self, parent: _Node, i: int) -> None:
        """Repair parent.children[i], which has underflowed, by borrowing
        from a neighbour or merging with one."""
        child = parent.children[i]
        left = parent.children[i - 1] if i > 0 else None
        right = parent.children[i + 1] if i < len(parent.children) - 1 else None
        spare = self.min_leaf if child.leaf else self.min_internal

        if left is not None and len(left.keys) > spare:
            self._borrow_from_left(parent, i)
        elif right is not None and len(right.keys) > spare:
            self._borrow_from_right(parent, i)
        elif left is not None:
            self._merge(parent, i - 1)   # merge left + child
        else:
            self._merge(parent, i)       # merge child + right

    def _borrow_from_left(self, parent: _Node, i: int) -> None:
        child, left = parent.children[i], parent.children[i - 1]
        if child.leaf:
            child.keys.insert(0, left.keys.pop())
            child.values.insert(0, left.values.pop())
            parent.keys[i - 1] = child.keys[0]           # new smallest on the right
        else:
            child.keys.insert(0, parent.keys[i - 1])     # rotate through parent
            child.children.insert(0, left.children.pop())
            parent.keys[i - 1] = left.keys.pop()

    def _borrow_from_right(self, parent: _Node, i: int) -> None:
        child, right = parent.children[i], parent.children[i + 1]
        if child.leaf:
            child.keys.append(right.keys.pop(0))
            child.values.append(right.values.pop(0))
            parent.keys[i] = right.keys[0]               # right sibling's new smallest
        else:
            child.keys.append(parent.keys[i])            # rotate through parent
            child.children.append(right.children.pop(0))
            parent.keys[i] = right.keys.pop(0)

    def _merge(self, parent: _Node, j: int) -> None:
        """Merge parent.children[j] and parent.children[j+1] into one node,
        dropping separator parent.keys[j]."""
        left, right = parent.children[j], parent.children[j + 1]
        if left.leaf:
            left.keys.extend(right.keys)
            left.values.extend(right.values)
            left.next = right.next                       # re-link the leaf chain
        else:
            left.keys.append(parent.keys[j])             # pull separator down
            left.keys.extend(right.keys)
            left.children.extend(right.children)
        parent.keys.pop(j)
        parent.children.pop(j + 1)

    # ------------------------------------------------------------------ #
    # INSPECTION / DEBUG                                                  #
    # ------------------------------------------------------------------ #
    def __len__(self) -> int:
        return self._size

    def height(self) -> int:
        h, node = 1, self.root
        while not node.leaf:
            h += 1
            node = node.children[0]
        return h

    def levels(self) -> list:
        """Return keys per node, level by level (root first) — matches how the
        slides draw each tree."""
        out, cur = [], [self.root]
        while cur:
            out.append([list(n.keys) for n in cur])
            nxt = []
            for n in cur:
                if not n.leaf:
                    nxt.extend(n.children)
            cur = nxt
        return out

    def leaf_chain(self) -> list:
        """Return the leaves left-to-right by following the .next links."""
        node = self.root
        while not node.leaf:
            node = node.children[0]
        chain = []
        while node is not None:
            chain.append(list(node.keys))
            node = node.next
        return chain

    def pretty(self) -> str:
        lines = []
        for depth, level in enumerate(self.levels()):
            label = "root " if depth == 0 else ("leaves" if depth == len(self.levels()) - 1 else f"L{depth}   ")
            lines.append(f"{label}: " + "  ".join("[" + ",".join(map(str, ks)) + "]" for ks in level))
        lines.append("chain : " + " -> ".join("[" + ",".join(map(str, ks)) + "]" for ks in self.leaf_chain()))
        return "\n".join(lines)

    # ------------------------------------------------------------------ #
    # INVARIANT CHECK (for testing)                                      #
    # ------------------------------------------------------------------ #
    def check_invariants(self) -> None:
        depths = set()

        def walk(node, depth, lo, hi):
            # keys sorted and within (lo, hi]
            assert node.keys == sorted(node.keys), "keys not sorted"
            for k in node.keys:
                assert (lo is None or k >= lo) and (hi is None or k < hi), "key out of range"
            if node.leaf:
                depths.add(depth)
                if node is not self.root:
                    assert len(node.keys) >= self.min_leaf, f"leaf underfull: {node.keys}"
                assert len(node.keys) <= self.max_keys, f"leaf overfull: {node.keys}"
                return
            assert len(node.children) == len(node.keys) + 1, "k keys must have k+1 children"
            if node is not self.root:
                assert len(node.keys) >= self.min_internal, f"internal underfull: {node.keys}"
            assert len(node.keys) <= self.max_keys, f"internal overfull: {node.keys}"
            bounds = [lo] + node.keys + [hi]
            for idx, ch in enumerate(node.children):
                walk(ch, depth + 1, bounds[idx], bounds[idx + 1])

        walk(self.root, 0, None, None)
        assert len(depths) == 1, f"leaves at differing depths: {depths}"
        # leaf chain must be globally sorted and match items()
        chain_keys = []
        node = self.root
        while not node.leaf:
            node = node.children[0]
        while node is not None:
            chain_keys.extend(node.keys)
            node = node.next
        assert chain_keys == sorted(chain_keys), "leaf chain not globally sorted"


if __name__ == "__main__":
    # ---- Reproduce the worked example from the slides (insert 1..10) ----
    print("=" * 64)
    print("WORKED EXAMPLE  (max 3 keys/node)  — insert 1..10")
    print("=" * 64)
    t = BPlusTree(max_keys=3)
    narration = {
        3:  "Step 1: 1,2,3 fill a single leaf (also the root).",
        4:  "Step 2: insert 4 -> leaf splits, 3 COPIED up.",
        6:  "Step 3: insert 5,6 -> right leaf splits, 5 COPIED up.",
        8:  "Step 4: insert 7,8 -> split copies 7 up; root now full [3,5,7].",
        9:  "Step 5: insert 9 -> lands in last leaf, no split.",
        10: "Step 7: insert 10 -> leaf splits (9 copied up), root overflows and\n"
            "         splits, 7 MOVES up -> new root, height grows to 3.",
    }
    for k in range(1, 11):
        t.insert(k)
        t.check_invariants()
        if k in narration:
            print("\n" + narration[k])
            print(t.pretty())

    print("\n" + "=" * 64)
    print("SEARCH  (always ends at a leaf)")
    print("=" * 64)
    print("search(9)  ->", t.search(9), " (data found in leaf, not at the separator above)")
    print("search(11) ->", t.search(11))

    print("\n" + "=" * 64)
    print("RANGE SCAN  4 <= key <= 8  (descend once, then walk leaf links)")
    print("=" * 64)
    print([k for k, _ in t.range(4, 8)])

    print("\n" + "=" * 64)
    print("DELETION  (borrow / merge, cascading like a B-tree)")
    print("=" * 64)
    for k in (1, 2, 3):
        t.delete(k)
        t.check_invariants()
        print(f"\nafter delete({k}):")
        print(t.pretty())

WORKED EXAMPLE  (max 3 keys/node)  — insert 1..10

Step 1: 1,2,3 fill a single leaf (also the root).
root : [1,2,3]
chain : [1,2,3]

Step 2: insert 4 -> leaf splits, 3 COPIED up.
root : [3]
leaves: [1,2]  [3,4]
chain : [1,2] -> [3,4]

Step 3: insert 5,6 -> right leaf splits, 5 COPIED up.
root : [3,5]
leaves: [1,2]  [3,4]  [5,6]
chain : [1,2] -> [3,4] -> [5,6]

Step 4: insert 7,8 -> split copies 7 up; root now full [3,5,7].
root : [3,5,7]
leaves: [1,2]  [3,4]  [5,6]  [7,8]
chain : [1,2] -> [3,4] -> [5,6] -> [7,8]

Step 5: insert 9 -> lands in last leaf, no split.
root : [3,5,7]
leaves: [1,2]  [3,4]  [5,6]  [7,8,9]
chain : [1,2] -> [3,4] -> [5,6] -> [7,8,9]

Step 7: insert 10 -> leaf splits (9 copied up), root overflows and
         splits, 7 MOVES up -> new root, height grows to 3.
root : [7]
L1   : [3,5]  [9]
leaves: [1,2]  [3,4]  [5,6]  [7,8]  [9,10]
chain : [1,2] -> [3,4] -> [5,6] -> [7,8] -> [9,10]

SEARCH  (always ends at a leaf)
search(9)  -> 9  (data found in leaf, not at the sep

In [2]:


from __future__ import annotations
from bisect import bisect_left, bisect_right
from typing import Any, Iterator, Optional


class _Node:
    """Leaves carry parallel `values` and a `next` link; internal nodes carry
    `children` and hold separator keys only (a pure index)."""
    __slots__ = ("leaf", "keys", "values", "children", "next")

    def __init__(self, leaf: bool):
        self.leaf = leaf
        self.keys: list = []
        self.values: list = []             # leaves only
        self.children: list["_Node"] = []  # internal only
        self.next: Optional["_Node"] = None  # leaf link


class BPlusTree:
    def __init__(self, max_keys: int = 3):
        if max_keys < 2:
            raise ValueError("max_keys must be >= 2")
        self.max_keys = max_keys
        # Minimum occupancy for NON-root nodes ("half full").
        self.min_leaf = (max_keys + 1) // 2            # max_keys=3 -> 2
        self.min_internal = (max_keys + 2) // 2 - 1    # max_keys=3 -> 1 (min 2 children)
        self.root = _Node(leaf=True)
        self._size = 0

    # ================================================================== #
    #  DELETION  (the focus of this module)                              #
    # ================================================================== #
    def delete(self, key) -> bool:
        """Remove `key` from the tree. Returns True iff it was present.

        Public entry point (slide 3): recursively delete, then collapse the
        root if a cascade has thinned it to a single child.
        """
        removed = self._delete(self.root, key)
        if not self.root.leaf and len(self.root.keys) == 0:
            # The root lost its last separator during a cascade — its single
            # remaining child becomes the new root and the tree height drops.
            self.root = self.root.children[0]
        return removed

    def _delete(self, node: _Node, key) -> bool:
        """Delete `key` from the subtree rooted at `node`. After recursing into
        a child, repair that child if it underflowed."""
        if node.leaf:
            # ---- Step 1: remove at the leaf (slide 3) --------------------
            i = bisect_left(node.keys, key)
            if i < len(node.keys) and node.keys[i] == key:
                node.keys.pop(i)
                node.values.pop(i)
                self._size -= 1
                return True
            return False  # key absent

        # Route to the child that owns the key ("equals go right").
        i = bisect_right(node.keys, key)
        removed = self._delete(node.children[i], key)
        # ---- Steps 2-4: repair the child if the deletion underflowed it ---
        if removed and self._is_underfull(node.children[i]):
            self._repair(node, i)
        return removed

    def _is_underfull(self, node: _Node) -> bool:
        limit = self.min_leaf if node.leaf else self.min_internal
        return len(node.keys) < limit

    def _repair(self, parent: _Node, i: int) -> None:
        """Fix `parent.children[i]`, which has underflowed, by borrowing from a
        neighbour with a spare key, or merging with one (slides 3-5)."""
        child = parent.children[i]
        left = parent.children[i - 1] if i > 0 else None
        right = parent.children[i + 1] if i < len(parent.children) - 1 else None
        spare = self.min_leaf if child.leaf else self.min_internal

        if left is not None and len(left.keys) > spare:
            self._borrow_from_left(parent, i)
        elif right is not None and len(right.keys) > spare:
            self._borrow_from_right(parent, i)
        elif left is not None:
            self._merge(parent, i - 1)   # merge  left + child
        else:
            self._merge(parent, i)       # merge  child + right

    # ---- BORROW (slide 4 for leaves; B-tree rotation for internals) ------ #
    def _borrow_from_left(self, parent: _Node, i: int) -> None:
        child, left = parent.children[i], parent.children[i - 1]
        if child.leaf:
            # Move the left leaf's largest key across as DATA, then reset the
            # separator to the new smallest key of the right leaf.
            child.keys.insert(0, left.keys.pop())
            child.values.insert(0, left.values.pop())
            parent.keys[i - 1] = child.keys[0]
        else:
            # Internal borrow == ordinary B-tree rotation THROUGH the parent.
            child.keys.insert(0, parent.keys[i - 1])
            child.children.insert(0, left.children.pop())
            parent.keys[i - 1] = left.keys.pop()

    def _borrow_from_right(self, parent: _Node, i: int) -> None:
        child, right = parent.children[i], parent.children[i + 1]
        if child.leaf:
            child.keys.append(right.keys.pop(0))
            child.values.append(right.values.pop(0))
            parent.keys[i] = right.keys[0]       # right sibling's new smallest
        else:
            child.keys.append(parent.keys[i])
            child.children.append(right.children.pop(0))
            parent.keys[i] = right.keys.pop(0)

    # ---- MERGE (slide 5 for leaves; B-tree merge for internals) ---------- #
    def _merge(self, parent: _Node, j: int) -> None:
        """Merge `parent.children[j]` and `parent.children[j+1]` into one node
        and drop separator `parent.keys[j]`."""
        left, right = parent.children[j], parent.children[j + 1]
        if left.leaf:
            # Concatenate the data; DROP the separator (its value already lives
            # in a leaf); re-link the chain past the vanished right leaf.
            left.keys.extend(right.keys)
            left.values.extend(right.values)
            left.next = right.next
        else:
            # Internal merge: the separator IS real routing structure here, so
            # it is pulled down between the two children's key lists.
            left.keys.append(parent.keys[j])
            left.keys.extend(right.keys)
            left.children.extend(right.children)
        parent.keys.pop(j)
        parent.children.pop(j + 1)

    # ================================================================== #
    #  SUPPORTING ENGINE  (compact — same behaviour as bplustree.py)     #
    # ================================================================== #
    def _find_leaf(self, key) -> _Node:
        node = self.root
        while not node.leaf:
            node = node.children[bisect_right(node.keys, key)]
        return node

    def search(self, key) -> Optional[Any]:
        leaf = self._find_leaf(key)
        i = bisect_left(leaf.keys, key)
        return leaf.values[i] if i < len(leaf.keys) and leaf.keys[i] == key else None

    def __contains__(self, key) -> bool:
        leaf = self._find_leaf(key)
        i = bisect_left(leaf.keys, key)
        return i < len(leaf.keys) and leaf.keys[i] == key

    def insert(self, key, value: Any = None) -> None:
        if value is None:
            value = key
        split = self._insert(self.root, key, value)
        if split is not None:
            sep, left, right = split
            r = _Node(leaf=False); r.keys = [sep]; r.children = [left, right]
            self.root = r

    def _insert(self, node, key, value):
        if node.leaf:
            i = bisect_left(node.keys, key)
            if i < len(node.keys) and node.keys[i] == key:
                node.values[i] = value; return None
            node.keys.insert(i, key); node.values.insert(i, value); self._size += 1
            if len(node.keys) > self.max_keys:
                mid = len(node.keys) // 2
                right = _Node(leaf=True)
                right.keys = node.keys[mid:]; right.values = node.values[mid:]
                node.keys = node.keys[:mid]; node.values = node.values[:mid]
                right.next = node.next; node.next = right
                return (right.keys[0], node, right)   # COPY up
            return None
        i = bisect_right(node.keys, key)
        split = self._insert(node.children[i], key, value)
        if split is None:
            return None
        sep, left, right = split
        node.keys.insert(i, sep); node.children[i] = left; node.children.insert(i + 1, right)
        if len(node.keys) > self.max_keys:
            mid = len(node.keys) // 2
            up = node.keys[mid]                       # MOVE up
            r = _Node(leaf=False)
            r.keys = node.keys[mid + 1:]; r.children = node.children[mid + 1:]
            node.keys = node.keys[:mid]; node.children = node.children[:mid + 1]
            return (up, node, r)
        return None

    def range(self, lo, hi) -> Iterator[tuple]:
        node = self._find_leaf(lo)
        i = bisect_left(node.keys, lo)
        while node is not None:
            while i < len(node.keys):
                if node.keys[i] > hi:
                    return
                yield (node.keys[i], node.values[i]); i += 1
            node = node.next; i = 0

    def items(self) -> Iterator[tuple]:
        node = self.root
        while not node.leaf:
            node = node.children[0]
        while node is not None:
            yield from zip(node.keys, node.values); node = node.next

    # ---- construction helper: build an exact tree from a layout --------- #
    @classmethod
    def build_from_structure(cls, separators_by_level, leaves, max_keys=3):
        """Build a tree directly from an explicit layout, so a lecture's given
        starting tree can be reproduced without hunting for an insert order.

        `leaves`: list of key-lists, left to right, e.g. [[1,2],[3,4],...]
        `separators_by_level`: internal separator-lists per level, TOP first,
            e.g. [[[7]], [[3,5],[9,11]]]  ->  root [7]; next level [3,5] and [9,11]
        """
        t = cls(max_keys=max_keys)
        leaf_nodes = []
        for ks in leaves:
            n = _Node(leaf=True); n.keys = list(ks); n.values = list(ks)
            leaf_nodes.append(n)
        for a, b in zip(leaf_nodes, leaf_nodes[1:]):
            a.next = b
        below = leaf_nodes
        for level in reversed(separators_by_level):
            nodes, idx = [], 0
            for seps in level:
                node = _Node(leaf=False); node.keys = list(seps)
                node.children = below[idx: idx + len(seps) + 1]
                idx += len(seps) + 1
                nodes.append(node)
            below = nodes
        assert len(below) == 1, "layout does not resolve to a single root"
        t.root = below[0]
        t._size = sum(len(n.keys) for n in leaf_nodes)
        t.check_invariants()
        return t

    # ---- inspection / debug -------------------------------------------- #
    def __len__(self): return self._size

    def height(self):
        h, n = 1, self.root
        while not n.leaf:
            h += 1; n = n.children[0]
        return h

    def levels(self):
        out, cur = [], [self.root]
        while cur:
            out.append([list(n.keys) for n in cur])
            nxt = []
            for n in cur:
                if not n.leaf:
                    nxt.extend(n.children)
            cur = nxt
        return out

    def leaf_chain(self):
        n = self.root
        while not n.leaf:
            n = n.children[0]
        out = []
        while n is not None:
            out.append(list(n.keys)); n = n.next
        return out

    def pretty(self):
        lv = self.levels(); lines = []
        for d, level in enumerate(lv):
            tag = "root " if d == 0 else ("leaves" if d == len(lv) - 1 else f"L{d}   ")
            lines.append(f"{tag}: " + "  ".join("[" + ",".join(map(str, k)) + "]" for k in level))
        lines.append("chain : " + " -> ".join("[" + ",".join(map(str, k)) + "]" for k in self.leaf_chain()))
        return "\n".join(lines)

    def check_invariants(self):
        depths = set()
        def walk(node, depth, lo, hi):
            assert node.keys == sorted(node.keys), "keys unsorted"
            for k in node.keys:
                assert (lo is None or k >= lo) and (hi is None or k < hi), "key out of range"
            if node.leaf:
                depths.add(depth)
                if node is not self.root:
                    assert len(node.keys) >= self.min_leaf, f"leaf underfull {node.keys}"
                assert len(node.keys) <= self.max_keys, f"leaf overfull {node.keys}"
                return
            assert len(node.children) == len(node.keys) + 1, "k keys need k+1 children"
            if node is not self.root:
                assert len(node.keys) >= self.min_internal, f"internal underfull {node.keys}"
            assert len(node.keys) <= self.max_keys, f"internal overfull {node.keys}"
            b = [lo] + node.keys + [hi]
            for idx, ch in enumerate(node.children):
                walk(ch, depth + 1, b[idx], b[idx + 1])
        walk(self.root, 0, None, None)
        assert len(depths) == 1, f"leaves at differing depths {depths}"
        chain = [k for leaf in self.leaf_chain() for k in leaf]
        assert chain == sorted(chain), "leaf chain not globally sorted"


# ====================================================================== #
#  DEMOS — reproduce the deletion deck                                    #
# ====================================================================== #
def _worked_example():
    print("=" * 66)
    print("WORKED EXAMPLE  —  delete 12, 1, 6, 2   (max 3 keys/node, leaf min 2)")
    print("=" * 66)
    # Starting tree from slide 7.
    t = BPlusTree.build_from_structure(
        separators_by_level=[[[7]], [[3, 5], [9, 11]]],
        leaves=[[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12, 13]],
        max_keys=3,
    )
    print("start (slide 7):"); print(t.pretty())
    notes = {
        12: "delete 12 -> [11,12,13] has a spare; remove directly (no underflow)",
        1:  "delete 1  -> [1,2]->[2] underflows; sibling minimal -> MERGE, drop sep 3",
        6:  "delete 6  -> [5,6]->[5] underflows; left rich -> BORROW 4, reset sep 5->4",
        2:  "delete 2  -> [2,3]->[3] underflows; MERGE empties internal -> CASCADE (borrow)",
    }
    for k in (12, 1, 6, 2):
        t.delete(k); t.check_invariants()
        print("\n" + notes[k]); print(t.pretty())
    print("\nfinal leaf scan:", [k for k, _ in t.items()])


def _concept_demos():
    print("\n" + "=" * 66)
    print("CONCEPT DEMOS")
    print("=" * 66)

    # Slide 6: a separator can outlive its data.
    t = BPlusTree.build_from_structure(
        separators_by_level=[[[7]], [[3, 5], [9, 11]]],
        leaves=[[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12, 13]],
    )
    t.delete(11)
    print("\nSeparator outlives data (slide 6): delete 11")
    print(t.pretty())
    print(f"  separator 11 still routes; 11 in tree? {11 in t}; search(11) -> {t.search(11)}")

    # Slide 4: leaf borrow in isolation.
    t = BPlusTree.build_from_structure(separators_by_level=[[[5]]],
                                       leaves=[[2, 3, 4], [5, 6]])
    t.delete(6)  # [5,6]->[5] underflows, borrow 4 from left, separator 5->4
    print("\nBorrow between leaves (slide 4): after delete(6)")
    print(t.pretty(), "  <- separator reset to 4")

    # Slide 5: leaf merge in isolation.
    t = BPlusTree.build_from_structure(separators_by_level=[[[3, 5]]],
                                       leaves=[[1, 2], [3, 4], [5, 6]])
    t.delete(1)  # [1,2]->[2] underflows, merge with [3,4], drop separator 3
    print("\nMerge two leaves (slide 5): after delete(1)")
    print(t.pretty(), "  <- separator 3 dropped")


if __name__ == "__main__":
    _worked_example()
    _concept_demos()

WORKED EXAMPLE  —  delete 12, 1, 6, 2   (max 3 keys/node, leaf min 2)
start (slide 7):
root : [7]
L1   : [3,5]  [9,11]
leaves: [1,2]  [3,4]  [5,6]  [7,8]  [9,10]  [11,12,13]
chain : [1,2] -> [3,4] -> [5,6] -> [7,8] -> [9,10] -> [11,12,13]

delete 12 -> [11,12,13] has a spare; remove directly (no underflow)
root : [7]
L1   : [3,5]  [9,11]
leaves: [1,2]  [3,4]  [5,6]  [7,8]  [9,10]  [11,13]
chain : [1,2] -> [3,4] -> [5,6] -> [7,8] -> [9,10] -> [11,13]

delete 1  -> [1,2]->[2] underflows; sibling minimal -> MERGE, drop sep 3
root : [7]
L1   : [5]  [9,11]
leaves: [2,3,4]  [5,6]  [7,8]  [9,10]  [11,13]
chain : [2,3,4] -> [5,6] -> [7,8] -> [9,10] -> [11,13]

delete 6  -> [5,6]->[5] underflows; left rich -> BORROW 4, reset sep 5->4
root : [7]
L1   : [4]  [9,11]
leaves: [2,3]  [4,5]  [7,8]  [9,10]  [11,13]
chain : [2,3] -> [4,5] -> [7,8] -> [9,10] -> [11,13]

delete 2  -> [2,3]->[3] underflows; MERGE empties internal -> CASCADE (borrow)
root : [9]
L1   : [7]  [11]
leaves: [3,4,5]  [7,8]  [9,10